[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pajola/secai-course/blob/main/module-01-adversarial-ml/02_adversarial_text.ipynb)

# 2 · Adversarial Examples on Text — typos, triggers, transfer, defenses

**SecAI course · Adversarial Machine Learning, part 2**

Images live in a continuous space: "move every pixel by ε" is a natural perturbation. Text does not:

* the input is **discrete** (tokens) — there is no gradient *into* the input, only into embeddings;
* the perturbation must preserve **meaning** (for the human) and **plausibility** (not look obviously broken);
* so attacks work at the **character** level (typos, homoglyphs), the **word** level (synonyms, deletions), or the **sentence** level (paraphrase, appended "triggers").

We attack a sentiment classifier on SST-2 (short movie-review sentences). The victim is deliberately simple — TF-IDF + logistic regression — so that every step is inspectable; the same ideas (TextBugger, TextFooler, HotFlip, universal triggers) apply to transformers.

| | Evasion attack on text |
|---|---|
| **Goal** | flip the predicted sentiment |
| **Knowledge** | white-box (weights) → then score-based black-box (only the output probabilities) |
| **Capability** | edit at most *k* words; each edit must keep the sentence readable |

In [ ]:
# Setup — on Google Colab, fetch the course helper module (all other packages are preinstalled).
import os, sys, urllib.request
if "google.colab" in sys.modules and not os.path.exists("secai_utils.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/pajola/secai-course/main/module-01-adversarial-ml/secai_utils.py", "secai_utils.py")

In [1]:
import numpy as np, matplotlib.pyplot as plt, re, random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from secai_utils import load_sst2

random.seed(0); np.random.seed(0)
train_texts, y_train, test_texts, y_test = load_sst2()
print(len(train_texts), "train /", len(test_texts), "test sentences")
for t, l in list(zip(test_texts, y_test))[:4]:
    print(f"[{'POS' if l else 'NEG'}] {t}")

6920 train / 1821 test sentences
[NEG] no movement , no yuks , not much of anything
[NEG] a gob of drivel so sickly sweet , even the eager consumers of moore 's pasteurized ditties will retch it up like rancid cr me br l e
[NEG] gangs of new york is an unapologetic mess , whose only saving grace is that it ends by blowing just about everything up
[NEG] we never really feel involved with the story , as all of its ideas remain just that abstract ideas


## 2.1 The victim

In [2]:
vec = TfidfVectorizer(lowercase=True, token_pattern=r"[a-z']+")
clf = LogisticRegression(C=3.0, max_iter=2000)
victim = make_pipeline(vec, clf).fit(train_texts, y_train)

def acc(model, texts, labels): return (model.predict(texts) == labels).mean()
print(f"clean test accuracy: {acc(victim, test_texts, y_test):.3f}")

vocab = np.array(vec.get_feature_names_out())
coef = clf.coef_[0]
print("most positive words:", list(vocab[np.argsort(coef)[-10:]][::-1]))
print("most negative words:", list(vocab[np.argsort(coef)[:10]]))

clean test accuracy: 0.815
most positive words: ['fun', 'best', 'solid', 'powerful', 'entertaining', 'and', 'works', 'enjoyable', 'love', 'heart']
most negative words: ['too', 'bad', "n't", 'worst', 'dull', 'feels', 'suffers', 'stupid', 'mess', 'clich']


## 2.2 Character-level attack (TextBugger-style)

Idea: **(1)** find the words that matter most for the current prediction, **(2)** corrupt them just enough that the tokenizer no longer recognises them (so they fall out of the vocabulary and contribute nothing), **(3)** stop as soon as the label flips.

Three "bugs" that humans read through without noticing: an adjacent-character **swap** (`terrible → terrilbe`), an **inserted** character (`terri-ble`), and a **homoglyph** — replace a Latin letter with a visually identical Unicode one (`a` → Cyrillic `а`). The last one is invisible to the eye and devastating for any system that matches strings.

In [3]:
HOMOGLYPHS = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "x": "х", "i": "і"}

def bug_swap(w):
    if len(w) < 4: return None
    i = random.randrange(1, len(w) - 2); return w[:i] + w[i+1] + w[i] + w[i+2:]
def bug_insert(w):
    if len(w) < 3: return None
    i = random.randrange(1, len(w) - 1); return w[:i] + "-" + w[i:]
def bug_homoglyph(w):
    for ch in w:
        if ch in HOMOGLYPHS: return w.replace(ch, HOMOGLYPHS[ch], 1)
    return None
BUGS = [bug_swap, bug_insert, bug_homoglyph]

def tokens(text): return text.split()

def importance_whitebox(text, label):
    # contribution of each word to the *true* class  =  coef[word] * tfidf(word)   (sign-adjusted for class 0)
    x = vec.transform([text]); sign = 1 if label == 1 else -1
    contrib = {vocab[j]: sign * coef[j] * x[0, j] for j in x.indices}
    return [contrib.get(w.lower().strip("'"), 0.0) for w in tokens(text)]

def importance_blackbox(model, text, label):
    # leave-one-out: how much does P(true class) drop if we delete the word?  (only needs model.predict_proba)
    toks = tokens(text); p0 = model.predict_proba([text])[0, label]
    return [p0 - model.predict_proba([" ".join(toks[:i] + toks[i+1:])])[0, label] for i in range(len(toks))]

def attack_text(model, text, label, importance, max_edits=3):
    toks = tokens(text)
    order = np.argsort(importance(text, label))[::-1]       # most important words first
    edits = 0
    for i in order:
        if edits >= max_edits or model.predict([" ".join(toks)])[0] != label:
            break
        candidates = [b(toks[i]) for b in BUGS]
        candidates = [c for c in candidates if c]
        if not candidates: continue
        # score-based choice: pick the bug that lowers P(true class) the most
        scores = [model.predict_proba([" ".join(toks[:i] + [c] + toks[i+1:])])[0, label] for c in candidates]
        toks[i] = candidates[int(np.argmin(scores))]; edits += 1
    adv = " ".join(toks)
    return adv, model.predict([adv])[0] != label, edits

# demo on a few correctly classified sentences
idx = [i for i in range(len(test_texts)) if victim.predict([test_texts[i]])[0] == y_test[i]]
for i in idx[:6]:
    adv, ok, n = attack_text(victim, test_texts[i], y_test[i], lambda t, l: importance_whitebox(t, l))
    print(f"[{'POS' if y_test[i] else 'NEG'}] {test_texts[i]}\n   -> {adv}\n   flipped={ok} with {n} edit(s)\n")

[NEG] no movement , no yuks , not much of anything
   -> nо movement , nо yuks , not much of anytihng
   flipped=False with 3 edit(s)

[NEG] a gob of drivel so sickly sweet , even the eager consumers of moore 's pasteurized ditties will retch it up like rancid cr me br l e
   -> a gob of dirvel so sickly sweet , even the eager consumers of moore 's pasteurized ditties will retch it up like rancid cr mе br l e
   flipped=True with 2 edit(s)

[NEG] gangs of new york is an unapologetic mess , whose only saving grace is that it ends by blowing just about everything up
   -> gangs of new york is an unapologetic mses , whose olny saving grace is that it ends by blowing just about everything up
   flipped=True with 2 edit(s)

[NEG] we never really feel involved with the story , as all of its ideas remain just that abstract ideas
   -> we never re-ally feel involved with the story , as all of its ideas remain ju-st that abstract ideas
   flipped=True with 2 edit(s)

[POS] this is one of polans

Look closely at the homoglyph edits: the adversarial sentence is **byte-different but pixel-identical** to the original. (This is why input *sanitisation* — Unicode normalisation, confusable detection — is a security control, not a text-cleaning nicety.)

Now the success rate over the test set, with white-box vs. black-box word importance.

In [4]:
def attack_success_rate(model, importance, n=300, max_edits=3, seed=0):
    random.seed(seed)
    sample = idx[:n]; flips, total_edits = 0, 0
    advs = []
    for i in sample:
        adv, ok, k = attack_text(model, test_texts[i], y_test[i], importance, max_edits)
        flips += ok; total_edits += k; advs.append(adv)
    return flips / len(sample), total_edits / len(sample), advs

asr_wb, edits_wb, advs_wb = attack_success_rate(victim, importance_whitebox)
asr_bb, edits_bb, advs_bb = attack_success_rate(victim, lambda t, l: importance_blackbox(victim, t, l))
print(f"white-box importance:  success rate = {asr_wb:.2f}   avg edits = {edits_wb:.2f}  (max 3)")
print(f"black-box importance:  success rate = {asr_bb:.2f}   avg edits = {edits_bb:.2f}  (max 3)")

white-box importance:  success rate = 0.68   avg edits = 2.29  (max 3)
black-box importance:  success rate = 0.67   avg edits = 2.27  (max 3)


Three typos flip the majority of predictions — and the black-box attacker, who only sees output probabilities, does about as well as the one with the weights (the "which word matters" question is cheap to answer by querying).

## 2.3 Universal adversarial triggers (Wallace et al., 2019)

Instead of crafting a perturbation per input, find **one short string** that, prepended to *any* negative review, makes the model say "positive". For a linear model the recipe is embarrassingly simple: take the words with the largest positive weights. For neural models the same thing is done with gradient-guided search over the embedding space.

In [5]:
neg_idx = [i for i in idx if y_test[i] == 0]
neg_texts = [test_texts[i] for i in neg_idx]

for k in [1, 2, 3, 5]:
    trigger = " ".join(vocab[np.argsort(coef)[-k:]][::-1])
    flipped = (victim.predict([trigger + " " + t for t in neg_texts]) == 1).mean()
    print(f"trigger = {trigger!r:40s}  ->  {flipped:.0%} of negative reviews now 'positive'")

print("\nexample:", neg_texts[0], "\n   ->", trigger + " " + neg_texts[0])

trigger = 'fun'                                     ->  49% of negative reviews now 'positive'
trigger = 'fun best'                                ->  81% of negative reviews now 'positive'
trigger = 'fun best solid'                          ->  96% of negative reviews now 'positive'
trigger = 'fun best solid powerful entertaining'    ->  100% of negative reviews now 'positive'

example: no movement , no yuks , not much of anything 
   -> fun best solid powerful entertaining no movement , no yuks , not much of anything


Three words are enough to flip almost every negative review. The modern descendant of this attack is **prompt injection** against LLMs: an input-independent string that hijacks the model's behaviour regardless of the surrounding content.

## 2.4 Transferability

Same experiment as for images: craft the typo attacks on the LR victim, then feed them to models the attacker never saw.

In [6]:
models = {
    "LR word TF-IDF (victim)":       victim,
    "Naive Bayes, word counts":      make_pipeline(TfidfVectorizer(token_pattern=r"[a-z']+", use_idf=False, norm=None), MultinomialNB()).fit(train_texts, y_train),
    "LR, word+bigram TF-IDF":        make_pipeline(TfidfVectorizer(token_pattern=r"[a-z']+", ngram_range=(1, 2)), LogisticRegression(C=3, max_iter=2000)).fit(train_texts, y_train),
    "LR, char 2-5-gram TF-IDF":      make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)), LogisticRegression(C=3, max_iter=2000)).fit(train_texts, y_train),
}
sample_labels = y_test[idx[:300]]
clean_sample  = [test_texts[i] for i in idx[:300]]
print("(evaluated on 300 test sentences the victim classifies correctly, hence its clean accuracy of 1.0)")
print(f"{'model':32s} {'clean acc':>10s} {'acc on transferred adv':>24s}")
for name, m in models.items():
    print(f"{name:32s} {acc(m, clean_sample, sample_labels):10.3f} {acc(m, advs_wb, sample_labels):24.3f}")

(evaluated on 300 test sentences the victim classifies correctly, hence its clean accuracy of 1.0)
model                             clean acc   acc on transferred adv
LR word TF-IDF (victim)               1.000                    0.320
Naive Bayes, word counts              0.940                    0.563
LR, word+bigram TF-IDF                0.963                    0.577
LR, char 2-5-gram TF-IDF              0.943                    0.730


The typos were chosen against the word-level LR, yet they also hurt Naive Bayes and the bigram model: anything that tokenises on whitespace loses the same words. The **character n-gram** model is the odd one out — a hint for the defender.

## 2.5 Defenses

1. **Robust features** — character n-grams see `terri-ble` and `terrible` as mostly the same thing (already visible above).
2. **Input normalisation** — map confusable Unicode to ASCII (NFKC + a confusables table) and spell-correct out-of-vocabulary words back to the nearest vocabulary word. Cheap, and it kills the homoglyph trick outright.
3. **Adversarial data augmentation** — train on bugged copies of the training sentences so the model learns the bugged forms too.

Each is evaluated with the attacker **re-running** against the defended model (adaptive), not just replaying old adversarial examples.

In [7]:
import unicodedata, difflib

REVERSE_HOMOGLYPHS = {v: k for k, v in HOMOGLYPHS.items()}
train_vocab = set(vec.get_feature_names_out())
_by_key = {}                                            # index the vocabulary by (first letter, length) to keep spell-correction fast
for w in train_vocab:
    for dl in (-1, 0, 1): _by_key.setdefault((w[0], len(w) + dl), []).append(w)
_cache = {}
def spell_correct(lw):
    if lw not in _cache:
        close = difflib.get_close_matches(lw, _by_key.get((lw[0], len(lw)), []), n=1, cutoff=0.8)
        _cache[lw] = close[0] if close else lw
    return _cache[lw]

def normalise(text):
    text = unicodedata.normalize("NFKC", text)
    text = "".join(REVERSE_HOMOGLYPHS.get(c, c) for c in text).replace("-", "")
    out = []
    for w in text.split():
        lw = w.lower()
        if lw in train_vocab or not lw.isalpha(): out.append(w); continue
        out.append(spell_correct(lw))
    return " ".join(out)

class Normalised:
    # wrap any sklearn pipeline with an input-sanitisation step
    def __init__(self, model): self.model = model
    def predict(self, texts): return self.model.predict([normalise(t) for t in texts])
    def predict_proba(self, texts): return self.model.predict_proba([normalise(t) for t in texts])

def augment(texts, labels, n_bugs=2, seed=0):
    random.seed(seed); out_t, out_l = list(texts), list(labels)
    for t, l in zip(texts, labels):
        toks = t.split()
        for i in random.sample(range(len(toks)), min(n_bugs, len(toks))):
            b = random.choice(BUGS)(toks[i]); toks[i] = b or toks[i]
        out_t.append(" ".join(toks)); out_l.append(l)
    return out_t, np.array(out_l)

aug_texts, aug_labels = augment(train_texts, y_train)
defended = {
    "no defense (victim)":           victim,
    "char n-gram features":          models["LR, char 2-5-gram TF-IDF"],
    "input normalisation":           Normalised(victim),
    "adversarial augmentation":      make_pipeline(TfidfVectorizer(token_pattern=r"[a-z']+"), LogisticRegression(C=3, max_iter=2000)).fit(aug_texts, aug_labels),
    "normalisation + char n-grams":  Normalised(models["LR, char 2-5-gram TF-IDF"]),
}
print(f"{'defense':32s} {'clean acc':>10s} {'adaptive attack success':>26s}")
for name, m in defended.items():
    asr, _, _ = attack_success_rate(m, lambda t, l: importance_blackbox(m, t, l), n=200)
    print(f"{name:32s} {acc(m, test_texts, y_test):10.3f} {asr:26.2f}")

defense                           clean acc    adaptive attack success


no defense (victim)                   0.815                       0.67


char n-gram features                  0.823                       0.44


input normalisation                   0.815                       0.16


adversarial augmentation              0.813                       0.66


normalisation + char n-grams          0.819                       0.17


Two lessons hide in this table:

* **Random augmentation ≠ adversarial training.** Training on randomly bugged copies barely moves the needle, because the attacker does not pick *random* bugs — it picks the worst one for each word. Robustness comes from training on *worst-case* perturbations (as in notebook 1), not from noise.
* **Fixing the representation beats fixing the classifier.** Normalisation (undo confusables, spell-correct back into the vocabulary) removes most of the attack surface at zero accuracy cost; character n-grams help for the same reason.

No single defense is complete, and each costs something (compute, an engineering dependency on a vocabulary). Note that normalisation is a **non-differentiable preprocessing step** — exactly the kind of defense that *looked* unbreakable for images until adaptive attacks arrived; here the attacker's search is already black-box, so the number above is honest.

## Take-aways

* Text attacks trade the ε-ball for **readability constraints**; the attacker's budget is "edits a human would not notice".
* **Word importance is cheap to estimate by querying** → black-box attacks are nearly as strong as white-box ones.
* **Universal triggers** are input-agnostic and the ancestor of prompt injection.
* Adversarial text **transfers** across models that share a tokenisation; changing the *representation* (char n-grams) helps more than changing the classifier.
* Sanitise your inputs (Unicode normalisation, confusables) — it is the text equivalent of input validation.

### References
* Li et al., *TextBugger: Generating Adversarial Text Against Real-world Applications*, NDSS 2019.
* Ebrahimi et al., *HotFlip: White-Box Adversarial Examples for Text Classification*, ACL 2018.
* Jin et al., *Is BERT Really Robust? (TextFooler)*, AAAI 2020.
* Wallace et al., *Universal Adversarial Triggers for Attacking and Analyzing NLP*, EMNLP 2019.
* Boucher et al., *Bad Characters: Imperceptible NLP Attacks*, IEEE S&P 2022 (homoglyphs, invisible characters).